In [ ]:
#################################
# 4. Test
#################################

test_ds = NpDataset(test_x_path, test_y_path)
test_loader = DataLoader(
    test_ds,
    batch_size = batch_size,
    shuffle=False,
    num_workers_workers
)

model.eval()

correct = 0
total = 0

num_classes = 5

conf_mat = torch.zeros(num_classes, num_classes)

with torch.no_grad():
    for data in testloader:
        input, labels = data
        input = input.to(device)
        labels = labels.to(device)

        logits = model(x)
        pred = torch.argmax(logits, dim=1)

        total += y.size(0)
        correct +=(prid == y).sum().item()

        y_cpu = y.detach().cpu()
        pred_cpu = pred.detach().cpu()
        for t,p in zip(y_cpu.view(-1), pred_cpu.view(-1)):
            conf_mat[t.long(), p.long()] += 1

accuracy = correct / total

tp = conf_mat.diag()
fp = conf_mat.sum(0) - tp
fn = conf_mat.sum(1) - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)


precision_macro = precision_per_class.mean().item()
recall_macro    = recall_per_class.mean().item()
f1_macro        = f1_per_class.mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro): {precision_macro:.4f}")
print(f"Recall(macro)   : {recall_macro:.4f}")
print(f"F1(macro)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat.int())

# Class-wise accuracy (per attack)
#  - 정답 클래스별로 맞춘 비율 = TP / (TP+FN) = conf_mat[i,i] / row_sum(i)
# =========================
row_sum = conf_mat.sum(dim=1)          # 각 정답 클래스 샘플 수
tp = conf_mat.diag()                   # 각 클래스 TP
class_acc = tp / (row_sum + 1e-12)     # 클래스별 정확도(= recall)

print("\n=== Per-class (Attack) Accuracy ===")
for i in range(num_classes):
    n = int(row_sum[i].item())
    acc_i = 100.0 * class_acc[i].item()

    # 클래스 이름 매핑이 있으면 더 보기 좋음 (없으면 i만 출력)
    # 예: LABEL_NAME = {0:"Benign", 1:"Dos", 2:"Fuzzing", 3:"Replay", 4:"Spoofing"}
    name = LABEL_NAME[i] if "LABEL_NAME" in globals() and i in LABEL_NAME else f"class_{i}"

    print(f"{name:>10s} : {acc_i:6.2f}%  (correct {int(tp[i].item())}/{n})")